In [4]:
#All packages needed to run TwINFER simulation and inference are listed here. 
#If any of them are not installed, please install them using pip or conda env.
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numba
import tqdm
import scipy
import seaborn
import os
import sys
import joblib
from itertools import product
from pathlib import Path
import json

In [3]:
path_to_simulations = "/projects/b1042/GoyalLab/jaekj/TWINFER/TwINFER/grnInference/simulation_data/test/figure_3_simulations/"
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"
path_to_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_3_28012026/"
os.makedirs(path_to_plot_data, exist_ok=True)

## Importing necessary functions and defining functions used

In [6]:
# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle
    # calculate_pairwise_gene_gene_correlation_matrix,
    # check_system_in_steady_state,
    # check_gene_gene_correlation_threshold,
    # calculate_twin_random_pair_correlations,
    # differentiate_single_state_reg_and_multiple_states,
    # identify_reg_if_multiple_states,
    # get_cross_correlations,
    # identify_actual_directed_edges
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    split_and_merge_simulations,
    get_param_data, 
    plot_matrix_as_heatmap,
    print_summary,
    plot_network
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import spearmanr

#Functions to calculate cross-correlations for multiple replicates. Same idea as step 3 of infer_with_twinfer pipeline - rewritten here for optimizing the input
# and output for this specific use-case

# ==========================================
# CORRELATION FUNCTIONS
# ==========================================
def compute_correlation_matrix_fast(gene_matrix_t1, gene_matrix_t2, gene_list, 
                                   gene_pairs=None, threshold=None):
    """Fast correlation computation using pure NumPy operations."""
    n_genes = len(gene_list)
    gene_to_idx = {gene: i for i, gene in enumerate(gene_list)}
    
    # Initialize matrices
    raw_matrix = np.zeros((n_genes, n_genes))
    
    # Determine which pairs to compute
    if gene_pairs is None:
        pairs_to_compute = [(i, j) for i in range(n_genes) for j in range(n_genes)]
    else:
        pairs_to_compute = []
        for gene_1, gene_2 in gene_pairs:
            if gene_1 in gene_to_idx and gene_2 in gene_to_idx:
                i, j = gene_to_idx[gene_1], gene_to_idx[gene_2]
                pairs_to_compute.append((i, j))
    
    # Compute correlations for specified pairs
    for i, j in pairs_to_compute:
        corr = spearmanr(gene_matrix_t1[i, :], gene_matrix_t2[j, :]).correlation
        if np.isnan(corr):
            corr = 0.0
        raw_matrix[i, j] = corr
    
    # Convert to DataFrames
    raw_df = pd.DataFrame(raw_matrix, index=gene_list, columns=gene_list)
    return {"raw_matrix": raw_df}

def single_cell_shuffle(gene_matrix_t1, gene_matrix_t2, gene_list, gene_pairs, threshold):
    """Perform a single cell shuffling iteration."""
    n_cells = gene_matrix_t1.shape[1]
    shuffled_indices = np.random.permutation(n_cells)
    shuffled_matrix_t2 = gene_matrix_t2[:, shuffled_indices]
    return compute_correlation_matrix_fast(gene_matrix_t1, shuffled_matrix_t2, gene_list, gene_pairs, threshold)

def prepare_twin_data(df, gene_list, t1=10, t2=20):
    """
    Prepare twin data from simulation CSV file.
    Based on your data loading process.
    """
    # Basic checks
    if 'clone_id' not in df.columns:
        raise ValueError("CSV must contain 'clone_id' column")
    if 'time_step' not in df.columns:
        raise ValueError("CSV must contain 'time_step' column")
    
    n_clones_simulation = df['clone_id'].nunique()
    
    # Shuffle all clone IDs (like in your code)
    np.random.seed(101010)
    clone_ids_shuffled = np.random.permutation(n_clones_simulation)
    
    # Split into ratios (like in your code)
    n1 = n2 = n_clones_simulation // 4
    across_t_clones = clone_ids_shuffled[n1 + n2:]
    
    # Get across-time twins (like in your code)
    across_t_twin1 = df[
        (df['clone_id'].isin(across_t_clones)) & 
        (df['time_step'] == t1) & 
        (df['replicate'] == 1)
    ].reset_index(drop=True)
    
    across_t_twin2 = df[
        (df['clone_id'].isin(across_t_clones)) & 
        (df['time_step'] == t2) & 
        (df['replicate'] == 2)
    ].reset_index(drop=True)
    
    return across_t_twin1, across_t_twin2

def analyze_single_file(file_path, gene_list, t1=1, t2=20, n_shuffles=10000, threshold=0.02):
    """
    Analyze a single CSV file to get correlations and thresholds.
    """
    
    print(f"  Analyzing {os.path.basename(file_path)}")
    
    # Load data
    df = pd.read_csv(file_path)
    
    # Prepare twin data
    rep_0_t1, rep_1_t2 = prepare_twin_data(df, gene_list, t1, t2)
    
    if len(rep_0_t1) == 0 or len(rep_1_t2) == 0:
        print(f"    Warning: No twin data found")
        return None
    
    # Extract gene matrices
    gene_matrix_t1 = []
    gene_matrix_t2 = []
    
    for gene in gene_list:
        # Look for gene columns
        gene_col_t1 = f"{gene}_mRNA" if f"{gene}_mRNA" in rep_0_t1.columns else None
        gene_col_t2 = f"{gene}_mRNA" if f"{gene}_mRNA" in rep_1_t2.columns else None
        
        if not gene_col_t1:
            matching_cols = [col for col in rep_0_t1.columns if gene in col and 'mRNA' in col]
            gene_col_t1 = matching_cols[0] if matching_cols else None
            
        if not gene_col_t2:
            matching_cols = [col for col in rep_1_t2.columns if gene in col and 'mRNA' in col]
            gene_col_t2 = matching_cols[0] if matching_cols else None
        
        if gene_col_t1 and gene_col_t2:
            gene_matrix_t1.append(rep_0_t1[gene_col_t1].values)
            gene_matrix_t2.append(rep_1_t2[gene_col_t2].values)
        else:
            print(f"    Warning: Could not find {gene} data")
            return None
    
    gene_matrix_t1 = np.array(gene_matrix_t1)
    gene_matrix_t2 = np.array(gene_matrix_t2)
    
    gene_pairs = [('gene_1', 'gene_2'), ('gene_2', 'gene_1'), ('gene_1', 'gene_1'), ('gene_2', 'gene_2')]
    
    # Get actual correlations
    actual_results = compute_correlation_matrix_fast(gene_matrix_t1, gene_matrix_t2, gene_list, gene_pairs, None)
    actual_matrix = actual_results['raw_matrix']
    
    gene_1_to_2 = actual_matrix.loc['gene_1', 'gene_2']
    gene_2_to_1 = actual_matrix.loc['gene_2', 'gene_1']
    
    # Get shuffled correlations for thresholds
    shuffled_results = Parallel(n_jobs=-1, verbose=0)(
        delayed(single_cell_shuffle)(gene_matrix_t1, gene_matrix_t2, gene_list, gene_pairs, None)
        for _ in range(n_shuffles)
    )
    
    # Calculate thresholds
    shuffled_12 = []
    shuffled_21 = []
    
    for result in shuffled_results:
        shuffled_matrix = result['raw_matrix']
        shuffled_12.append((shuffled_matrix.loc['gene_1', 'gene_2']))
        shuffled_21.append((shuffled_matrix.loc['gene_2', 'gene_1']))

    # Calculate threshold for this pair: gene 1 -> gene 2
    p_plus_12 = np.mean(shuffled_12 >= gene_1_to_2)
    p_minus_12 = np.mean(shuffled_12 <= gene_1_to_2)
    p_value_12 = min(2 * p_plus_12, 2 * p_minus_12, 1.0)
    is_significant_12 = p_value_12 < threshold
    print(f"Observed correlation: {gene_1_to_2:.4f}")
    print(f"p-value: {p_value_12:.4f}")
    print(f"Significant at α={threshold}: {is_significant_12}")

    # Calculate threshold for this pair: gene 1 -> gene 2
    p_plus_21 = np.mean(shuffled_21 >= gene_2_to_1)
    p_minus_21 = np.mean(shuffled_21 <= gene_2_to_1)
    p_value_21 = min(2 * p_plus_21, 2 * p_minus_21, 1.0)
    is_significant_21 = p_value_21 < threshold
    print(f"Observed correlation: {gene_2_to_1:.4f}")
    print(f"p-value: {p_value_21:.4f}")
    print(f"Significant at α={threshold}: {is_significant_21}")
    
    threshold_12 = max(
        abs(np.percentile(shuffled_12, 1)),
        abs(np.percentile(shuffled_12, 99))
    )
    threshold_21 = max(
        abs(np.percentile(shuffled_21, 1)),
        abs(np.percentile(shuffled_21, 99))
    )

    combined_null = np.concatenate([shuffled_12, shuffled_21])
    threshold_combined = max(
        abs(np.percentile(combined_null, 1)),
        abs(np.percentile(combined_null, 99))
    )

    
    print(f"    Correlations: gene_1→gene_2={gene_1_to_2:.3f}, gene_2→gene_1={gene_2_to_1:.3f}")
    print(f"    Thresholds: gene_1→gene_2={threshold_12:.3f}, gene_2→gene_1={threshold_21:.3f}")
    
    return {
        'gene_1_to_gene_2': gene_1_to_2,
        'gene_2_to_gene_1': gene_2_to_1,
        'pvalue_12': p_value_12,
        'pvalue_21': p_value_21,
        'threshold_12': threshold_12,
        'threshold_21': threshold_21,
        'threshold_combined': threshold_combined
    }
# =========================================================================================================

def cross_correlation_data_across_replicates(base_path, folder_names, gene_list, t1=1, t2=20, n_shuffles=10000):
    """
    Collect correlations and thresholds from all files specified as this specific network type.
    """
    
    all_results = []
    
    for folder in folder_names:
        folder_path = os.path.join(base_path, folder)
        print(f"Processing {folder}...")
        
        if not os.path.exists(folder_path):
            print(f"  Folder not found: {folder_path}")
            continue
            
        # Find CSV files
        csv_files = [f for f in os.listdir(folder_path) if f.startswith('df_') and f.endswith('.csv') and ("0801" in f or "0901" in f)][:20]
        print(f"  Found {len(csv_files)} files")
        for csv_file in csv_files:
            file_path = os.path.join(folder_path, csv_file)
            print(csv_file)
            try:
                result = analyze_single_file(file_path, gene_list, t1, t2, n_shuffles)
                
                if result:
                    result['Condition'] = folder
                    result['File'] = csv_file
                    all_results.append(result)
                    
            except Exception as e:
                print(f"    Error with {csv_file}: {e}")
                continue
    
    return pd.DataFrame(all_results)

In [5]:
# Network types (folder names)
network_types = [
    "A_B",
    "A_rep_B", 
    "A_and_B_both_repress",
    "A_rep_B_B_to_A",
    "A_to_B",
    "A_and_B"
]

# Base configuration template
base_config_template = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 48,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_A_to_B.txt",  # Will be updated per network type
    "param_csv": f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,1]],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
}

# Time points for analysis in figure 3
t1, t2 = 1, 20  

In [ ]:
# Process each network type
all_correlation_matrices = {}

for network_type in network_types:
    print(f"Processing network type: {network_type}")
    
    # Find simulation file for this network type
    network_folder = os.path.join(path_to_simulations, network_type)
    if not os.path.exists(network_folder):
        print(f"Warning: Network folder not found: {network_folder}")
        continue
    
    # Look for CSV files in the network type folder
    csv_files = [f for f in Path(network_folder).glob("*.csv")]
    print(csv_files)
    if not csv_files:
        print(f"Warning: No CSV files starting with 'df' found in {network_folder}")
        continue
    
    # Use the first CSV file found (or you can add logic to select specific one)
    path_to_simulation_file = str(csv_files[0])
    print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
    # Update config for this network type
    config = base_config_template.copy()
    config["type"] = network_type
    config["log_file"] = f"{path_to_code_repo}/example_simulation_output/{network_type}_log.jsonl"
    
    # Check if required files exist
    if not os.path.exists(config["path_to_connectivity_matrix"]):
        print(f"Warning: Connectivity matrix not found for {network_type}")
        continue
    print(config["param_csv"])
    if not os.path.exists(config["param_csv"]):
        print(f"Warning: Parameter CSV not found for {network_type}")
        continue

    try:
        # Run inference for this network type
        correlation_matrices = infer_with_twinfer(
                path_to_simulation_file = path_to_simulation_file, 
                base_config = config, 
                t1 = t1, t2 = t2,
                check_for_steady_state=False, 
                plot_correlation_matrices_as_heatmap=False, 
                have_any_output=False,
                infer_direction_for_which_edges = "all-edges",
                merge_to_multiple_states  = False,
                match_sim_details = False
            )
            
        # Store the correlation matrices
        all_correlation_matrices[network_type] = correlation_matrices
            
        # Save the directional correlation matrix for this network type
        correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
        correlation_matrices['direction_matrix'].to_csv(correlation_file_name)            
        print(f"Successfully processed {network_type}")
        print(f"Saved correlation matrix to: {correlation_file_name}")
    except Exception as e:
        print(f"Error processing {network_type}: {str(e)}")
        continue

### Code for calculating cross-correlations across all replicates

In [15]:
# Parameters
sub_folder_names = ["A_B", "A_to_B", "A_and_B", "A_rep_B", "A_and_B_both_repress", "A_rep_B_B_to_A", ]
gene_list = ['gene_1', 'gene_2']
t1, t2 = 1, 20  # Adjust these to your time points
n_shuffles = 10000  # Number of shuffles for threshold calculation

# Collect all data
print("\n1. Collecting data from all files...")
df_results = cross_correlation_data_across_replicates(path_to_simulations, sub_folder_names, gene_list, t1, t2, n_shuffles)

if df_results.empty:
    print("No data collected!")
else:
    df_results.to_csv(f"{path_to_plot_data}/box_plot_data.csv")
print(f"Collected {len(df_results)} files from {df_results['Condition'].nunique()} conditions")

# Summary stats
print("\n4. Summary statistics:")
summary = df_results.groupby('Condition')[['gene_1_to_gene_2', 'gene_2_to_gene_1']].describe()
print(summary)


1. Collecting data from all files...
Processing A_B...
  Found 0 files
Processing A_to_B...
  Found 0 files
Processing A_and_B...
  Found 10 files
df_rows_0_0_08012026_224459_ncells_6000_A_and_B_rep_8_abf5a605.csv
  Analyzing df_rows_0_0_08012026_224459_ncells_6000_A_and_B_rep_8_abf5a605.csv
Observed correlation: 0.0337
p-value: 0.0548
Significant at α=0.02: False
Observed correlation: 0.1080
p-value: 0.0000
Significant at α=0.02: True
    Correlations: gene_1→gene_2=0.034, gene_2→gene_1=0.108
    Thresholds: gene_1→gene_2=0.042, gene_2→gene_1=0.043
df_rows_0_0_08012026_211444_ncells_6000_A_and_B_rep_5_f5055d94.csv
  Analyzing df_rows_0_0_08012026_211444_ncells_6000_A_and_B_rep_5_f5055d94.csv
Observed correlation: 0.0757
p-value: 0.0002
Significant at α=0.02: True
Observed correlation: 0.0852
p-value: 0.0000
Significant at α=0.02: True
    Correlations: gene_1→gene_2=0.076, gene_2→gene_1=0.085
    Thresholds: gene_1→gene_2=0.044, gene_2→gene_1=0.044
df_rows_0_0_08012026_231506_ncells_

## 5-gene cascade

In [ ]:
import os
import pandas as pd
from pathlib import Path
# Base configuration template
base_config_five_gene = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 24,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_5_gene_linear_cascade.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,0,0,0,0]],
    "path_to_plot_data": path_to_plot_data,
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18,
}
path_to_simulation_file = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_3_simulations/df_rows_0_0_0_0_0_03122025_221647_ncells_6000_Five_gene_cascade_fixed_K_0_7_6a281f3f.csv"


In [ ]:
twinfer_kwargs = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config_five_gene,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 10, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": "all-potential-edges", #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 16
}

In [ ]:
# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
# Update config for this network type
config = base_config_five_gene
network_type = "five_gene_linear_cascade" 
config["type"] = {network_type}

def make_json_safe(obj):
    if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
        return obj.to_dict()
    if isinstance(obj, set):
        return list(obj)
    return obj
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
           **twinfer_kwargs
        )
        
    # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
    # Save the directional correlation matrix for this network type
    json_safe = {
        k: make_json_safe(v)
        for k, v in correlation_matrices.items()
    }
    path_to_json_file = f"{path_to_plot_data}five_gene_linear_cascade_correlation_matrices.json"

    with open(path_to_json_file, "w") as f:
        json.dump(json_safe, f, indent=2)

    print(correlation_matrices.keys())
    correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
    gene_correlation_file_name = f"{path_to_plot_data}/gene_correlation_type_{network_type}.csv"
    correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
    correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
    correlation_matrices
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")

### Random cross-correlation

In [ ]:
twinfer_kwargs_random = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config_five_gene,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 10, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": "all-potential-regulation", #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 16,
    "remove_twin_structure": True
}

In [ ]:
# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
# Update config for this network type
config = base_config_five_gene
network_type = "five_gene_linear_cascade" 
config["type"] = {network_type}

def make_json_safe(obj):
    if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
        return obj.to_dict()
    if isinstance(obj, set):
        return list(obj)
    return obj
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
           **twinfer_kwargs_random
        )
        
    # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
    # Save the directional correlation matrix for this network type
    json_safe = {
        k: make_json_safe(v)
        for k, v in correlation_matrices.items()
    }
    path_to_json_file = f"{path_to_plot_data}five_gene_linear_cascade_correlation_matrices_random.json"

    with open(path_to_json_file, "w") as f:
        json.dump(json_safe, f, indent=2)

    print(correlation_matrices.keys())
    correlation_file_name = f"{path_to_plot_data}/filtered_directional_correlation_type_{network_type}_random.csv"
    gene_correlation_file_name = f"{path_to_plot_data}/gene_correlation_type_{network_type}_random.csv"
    correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
    correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
    correlation_matrices
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")

## Analyzing with grnBoost2 (Note: this needs a separate env set - details provided in grnBoost_environment.yaml)

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from scipy.stats import spearmanr, rankdata
from joblib import Parallel, delayed
import warnings
import gc
import argparse
from tqdm.auto import tqdm
import glob
import matplotlib.pyplot as plt
import networkx as nx
import re
import sys

In [2]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import precision_score, recall_score, f1_score
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, precision_score, recall_score, f1_score

In [ ]:
import os
import pandas as pd
from pathlib import Path
# Base configuration template
base_config_five_gene = {
    'n_cells': 6000,
    'simulation_time_before_division': 1000,
    'twin_simulation_time_after_division': 24,
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_5_gene_linear_cascade.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "rows_to_use": [[0,0,0,0,0]],
    "log_file": None,  # Will be updated per network type
    "type": None,  # Will be updated per network type
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 18
}

### first analyze the simulation files with TwINFER.

In [ ]:
def process_matrix(twinfer_kwargs, id):
    # Process each network type
    all_correlation_matrices = {}
    
    # Use the first CSV file found (or you can add logic to select specific one)        
    # Update config for this network type
    config = twinfer_kwargs['base_config']
    network_type = "five_gene_cascade" 

    def make_json_safe(obj):
        if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
            return obj.to_dict()
        if isinstance(obj, set):
            return list(obj)
        return obj
        
    # Check if required files exist
    if not os.path.exists(config["path_to_connectivity_matrix"]):
        print(f"Warning: Connectivity matrix not found for {network_type}")

    if not os.path.exists(config["param_csv"]):
        print(f"Warning: Parameter CSV not found for {network_type}")
    correlation_matrices = []
    try:
        print("running")
        # Run inference for this network type
        correlation_matrices = infer_with_twinfer(
            **twinfer_kwargs
            )
            

    except Exception as e:
        raise(f"Error processing {network_type}: {str(e)}")
    
    import json
    # Store the correlation matrices
    all_correlation_matrices[network_type] = correlation_matrices
        
    # Save the directional correlation matrix for this network type
    json_safe = {
        k: make_json_safe(v)
        for k, v in correlation_matrices.items()
    }
    path_to_json_file = f"{path_to_plot_data}/metrics_analysis_data/five_gene_correlation_matrices_{id}.json"
    with open(path_to_json_file, "w") as f:
        json.dump(json_safe, f, indent=2)
    print(correlation_matrices.keys())
    correlation_file_name = f"{path_to_plot_data}/metrics_analysis_data/five_gene_filtered_directional_correlation_type_{network_type}_{id}.csv"
    gene_correlation_file_name = f"{path_to_plot_data}/metrics_analysis_data/five_gene_gene_correlation_type_{network_type}_{id}.csv"
    correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
    correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
    correlation_matrices
    print(f"Successfully processed {network_type}")
    print(f"Saved correlation matrix to: {correlation_file_name}")
    return

In [ ]:
path_to_simulations = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_3_simulations"
prefix = "df_rows_0_0_0_0_0"

list_simulations = [
    f.path
    for f in os.scandir(path_to_simulations)
    if f.is_file() and f.name.startswith(prefix) and "rep" in f.name
]


In [ ]:
%%capture

for id, file_name in enumerate(list_simulations):
    twinfer_kwargs = {
    "path_to_simulation_file": file_name,
    "base_config": base_config_five_gene,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 12, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.02,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": 'all-potential-regulation', #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 18,
    "match_sim_details": False
    }
    process_matrix(twinfer_kwargs, id)

In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PATHS
# ============================================================
matrix_file = Path(f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_5_gene_linear_cascade.txt")
json_dir    = Path(f"{path_to_plot_data}/metrics_analysis_data/")          # contains edges_1.json, edges_2.json, ...
# plot_dir    = Path("path/to/output/plots")
# plot_dir.mkdir(parents=True, exist_ok=True)

json_files = sorted(json_dir.glob("five_gene_correlation*.json"))


# ============================================================
# LOAD BINARY MATRIX (ONCE)
# ============================================================
gt = pd.read_csv(matrix_file, header=None)
n = gt.shape[0]
genes = [f"gene_{i+1}" for i in range(n)]
gt.index = genes
gt.columns = genes

gene_labels = [f"g{i+1}" for i in range(n)]

from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
import seaborn as sns
from pathlib import Path

def make_reds_blues_colormap(vmin=-0.05, vmax=0.18):
    """Custom red–white–blue colormap with pure white at 0, asymmetric."""
    # Calculate where 0 falls in the range [vmin, vmax]
    zero_position = (0 - vmin) / (vmax - vmin)
    
    # Number of colors for each segment (proportional to range)
    n_total = 256
    n_reds = int(zero_position * n_total)  # colors from vmin to 0
    n_blues = n_total - n_reds  # colors from 0 to vmax
    
    # Calculate intensity based on actual distance from zero
    # For reds: map from vmin to 0, so max intensity at vmin
    red_intensity = abs(vmin) / max(abs(vmin), abs(vmax))  # 0.05/0.18 ≈ 0.28
    # For blues: map from 0 to vmax, so max intensity at vmax  
    blue_intensity = abs(vmax) / max(abs(vmin), abs(vmax))  # 0.18/0.18 = 1.0
    
    # Create color arrays with scaled intensities
    reds = plt.cm.Reds(np.linspace(0.8 * red_intensity, 0, n_reds))  # scaled dark to light red
    whites = np.ones((1, 4))  # pure white at 0
    blues = plt.cm.Blues(np.linspace(0, 0.8 * blue_intensity, n_blues))  # light to scaled dark blue
    
    colors = np.vstack((reds, whites, blues))
    return LinearSegmentedColormap.from_list('RedsBlues', colors)


metrics_five_gene_twinfer = []

# ============================================================
# MAIN LOOP
# ============================================================
for jf in json_files:

    # ---------------- load JSON ----------------
    with open(jf, "r") as f:
        cm = json.load(f)

    final_directed_edges = cm["final_directed_edges"]
    unfiltered_direction_matrix = pd.DataFrame(cm["unfiltered_direction_matrix"])
    multiple_states_and_reg = cm["gene_lists"]["multiple_states_and_reg"]
    all_gene_pairs = cm['all_gene_pairs']

    # enforce gene order
    unfiltered_direction_matrix.index = genes
    unfiltered_direction_matrix.columns = genes

    # ---------------- score ----------------
    # ============================================================
    # DIRECTED EDGE SCORING WITH ONE EXCEPTION (g2,g3)
    # ============================================================
    FP_edge = []
    FN_edge = []
    TP = FP = FN = TN = 0

    # quick lookup for predicted edges
    predicted_edges = set(tuple(e) for e in final_directed_edges)

    # special unordered exception pair
    EXCEPTION_PAIR = frozenset()
    can_be_2_states =  []

    for i, gi in enumerate(genes):
        for j, gj in enumerate(genes):
            if gi == gj:
                continue

            pair = frozenset([gi, gj])

            # ----------------------------------------------------
            # EXCEPTION: (gene_2, gene_3)
            # ----------------------------------------------------
            if pair == EXCEPTION_PAIR:
                # evaluate ONCE per unordered pair
                if gi > gj:
                    continue  # avoid double counting

                has_gi_gj = (gi, gj) in predicted_edges
                has_gj_gi = (gj, gi) in predicted_edges

                if (has_gi_gj and has_gj_gi) or (not has_gi_gj and not has_gj_gi):
                    if [gi, gj] in multiple_states_and_reg or [gj, gi] in multiple_states_and_reg:
                        print(gi, gj, multiple_states_and_reg)
                        continue
                    else:
                        FP += 2
                        FP_edge.append((gi,gj))
                        FP_edge.append((gj, gi))
                else:
                    FP += 1
                    FP_edge.append((gi,gj))
                continue

            # ----------------------------------------------------
            # NORMAL DIRECTED SCORING
            # ----------------------------------------------------
            gt_edge = gt.loc[gi, gj] == 1
            pred_edge = (gi, gj) in predicted_edges

            if gt_edge and pred_edge:
                if [gi, gj] in multiple_states_and_reg and [gi, gj] not in can_be_2_states:
                    FN += 1
                    FN_edge.append((gi,gj))
                else:
                    TP += 1
            elif gt_edge and not pred_edge:
                FN += 1
                FN_edge.append((gi,gj))
            elif not gt_edge and pred_edge:
                FP += 1
                FP_edge.append((gi,gj))
            else:
                TN += 1

    # ============================================================
    # METRICS
    # ============================================================
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    print(f"TP={TP}, FP={FP}, FN={FN}, TN={TN}")
    print(f"Precision={precision:.4f}")
    print(f"Recall={recall:.4f}")
    print(f"F1={f1:.4f}")
    print(f"FP: {FP_edge}")
    print(f"FN: {FN_edge}")

    metrics_five_gene_twinfer.append({
        "json": jf.name,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    #Plot TwINFER output
    gene_list = sorted(
        {g for pair in all_gene_pairs for g in pair},
        key=lambda x: int(x.split("_")[1])
    )
    gene_labels = [f"g{g.split('_')[1]}" for g in gene_list]
    # direction_matrix = unfiltered_direction_matrix.loc[gene_list, gene_list]
    data_matrix = unfiltered_direction_matrix.to_numpy(float)
    masked_matrix = np.fill_diagonal(data_matrix, np.nan)
    fig = plt.figure(figsize=(6,6))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.05, 0.95], hspace=0.1)
    cbar_ax = fig.add_subplot(gs[0])
    heatmap_ax = fig.add_subplot(gs[1])
    plot_matrix = data_matrix.copy()
    plot_matrix[:] = 0.0

    # Restore only final-ege correlations
    for g1, g2 in final_directed_edges:
        if g1 in gene_list and g2 in gene_list:
            i = gene_list.index(g1)
            j = gene_list.index(g2)
            plot_matrix[i, j] = data_matrix[i, j]
            plot_matrix[j, i] = data_matrix[j, i]
    np.fill_diagonal(plot_matrix, np.nan)
    vmin = np.nanmin(plot_matrix)
    vmax = np.nanmax(plot_matrix)
    cmap = make_reds_blues_colormap(vmin = vmin, vmax = vmax)

    cmap.set_bad(color="#D9D9D9")
    # --- draw heatmap ---
    sns.heatmap(
        plot_matrix,
        ax=heatmap_ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        square=True,
        cbar=True,
        cbar_ax=cbar_ax,
        cbar_kws={'orientation': 'horizontal'},
        linewidths=0.5,
        linecolor="black",
        # center=center
    )
    for g1, g2 in multiple_states_and_reg:
        if g1 in gene_list and g2 in gene_list:
            i = gene_list.index(g1)
            j = gene_list.index(g2)
            print(i, j)

            # Draw diagonal in cell (i, j) - top-left to bottom-right
            heatmap_ax.plot(
                [j, j+1],      # x: left → right
                [i, i+1],      # y: top → bottom
                linestyle="--",
                color="black",
                linewidth=1.5,
                clip_on=False
            )
            
            # Draw diagonal in symmetric cell (j, i) - top-left to bottom-right
            heatmap_ax.plot(
                [i, i+1],      # x: left → right
                [j, j+1],      # y: top → bottom
                linestyle="--",
                color="black",
                linewidth=1.5,
                clip_on=False
            )

    # --- labels ---
    cbar_ax.xaxis.set_label_position('top')
    cbar_ax.xaxis.tick_top()

    n = len(gene_labels)
    tick_pos = np.arange(n) + 0.5

    heatmap_ax.set_xticks(tick_pos)
    heatmap_ax.set_yticks(tick_pos)
    heatmap_ax.set_xticklabels(gene_labels, rotation=90, ha="center", va="top")
    heatmap_ax.set_yticklabels(gene_labels, rotation=0, ha="right", va="center")

    # keep limits tight to cells
    heatmap_ax.set_xlim(0, n)
    heatmap_ax.set_ylim(n, 0)

    # --- transparent background ---
    # plt.tight_layout()
    fig.patch.set_alpha(0)
    for ax in [heatmap_ax, cbar_ax]:
        ax.set_facecolor("none")
    for im in heatmap_ax.get_images() + cbar_ax.get_images():
        im.set_facecolor((1, 1, 1, 0))
        im.set_edgecolor((1, 1, 1, 0))
    for spine in heatmap_ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_edgecolor('black')
        spine.set_clip_on(False)

    plt.close()

# ============================================================
# METRICS SUMMARY
# ============================================================
metrics_df_five_gene_Twinfer = pd.DataFrame(metrics_five_gene_twinfer)
metrics_df_five_gene_Twinfer.to_csv(f"{path_to_plot_data}/metrics_analysis_data/metrics_TwINFER.csv")

### Analyzing with grnBoost

In [ ]:
%%capture
import os
# gene_names = [f"gene_{i}_mRNA" for i in np.arange(1,6)]
path_to_simulations = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_3_simulations/"
prefix = "df_"

list_simulations = [
    f.path
    for f in os.scandir(path_to_simulations)
    if f.is_file() and f.name.startswith(prefix) and "rep" in f.name
]
print(list_simulations)

for id, path in enumerate(list_simulations):
    df = pd.read_csv(path)
    df_5_gene_network = df[df['time_step'] == 1]
    gene_names = [f"gene_{i}_mRNA" for i in np.arange(1,6)]
    df_5_gene_network = df_5_gene_network[gene_names]
    X = df_5_gene_network.to_numpy(dtype=np.float64)

    tf_names = list(gene_names)

    network = grnboost2(
        expression_data=X,
        gene_names=tf_names,
        tf_names=tf_names,
        verbose=True,
        seed = 101010
    )
    network.to_csv(f"{path_to_plot_data}/metrics_analysis_data/grn_boost_five_gene_cascade_inference_{id}.csv")

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, precision_score, recall_score, f1_score

# ============================================================
# 1. Load ground-truth connectivity matrix
# ============================================================

gt_path = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/simulation_example_input_data/connectivity_matrix_5_gene_linear_cascade.txt"

# Load as numpy array
gt_matrix = np.loadtxt(gt_path, delimiter= ",")

n_genes = gt_matrix.shape[0]
genes = [f"gene_{i+1}_mRNA" for i in range(n_genes)]

gt_df = pd.DataFrame(
    gt_matrix,
    index=genes,
    columns=genes,
)

###SEARCH
# Binary ground truth: edge exists or not
gt_binary = (gt_df.values != 0).astype(int)
np.fill_diagonal(gt_binary, 0)

# ============================================================
# 2. Build predicted score matrix from network
# ============================================================
import os, pathlib
from pathlib import Path
network_files = Path(f"{path_to_plot_data}/metrics_analysis_data/")          # contains edges_1.json, edges_2.json, ...
list_network_files = sorted(network_files.glob("grn_boost_five_gene_cascade_inference_*.csv"))
metric_rows = []
for i, network_path in enumerate(list_network_files):
    # ---------------- load network ----------------
    network = pd.read_csv(network_path)

    # ---------------- dense prediction matrix ----------------
    pred_df = (
        network
        .pivot(index="TF", columns="target", values="importance")
        .reindex(index=genes, columns=genes, fill_value=0.0)
    )
    np.fill_diagonal(pred_df.values, 0.0)

    # ---------------- vectorize for metrics ----------------
    y_true = gt_binary.flatten()
    y_score = pred_df.values.flatten()

    mask = ~np.eye(len(genes), dtype=bool).flatten()
    y_true = y_true[mask]
    y_score = y_score[mask]

    # ========================================================
    # Best-F1 threshold (dense, comparable to GT)
    # ========================================================
    rows = []
    for t in np.unique(y_score)[::-1]:
        y_pred = (y_score >= t).astype(int)
        if y_pred.sum() == 0:
            continue

        rows.append({
            "threshold": t,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred),
            "n_edges": y_pred.sum(),
        })

    best = pd.DataFrame(rows).loc[lambda df: df["f1"].idxmax()]

    # ========================================================
    # GMM threshold (fit on predicted edges ONLY)
    # ========================================================
    x = network["importance"].values.reshape(-1, 1)
    MIN_POINTS = 2
    if len(x) >= MIN_POINTS and len(np.unique(x)) >= 2:
        gmm = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=0
        )
        gmm.fit(x)

        from scipy.stats import norm
        from scipy.optimize import brentq

        weights = gmm.weights_
        means = gmm.means_.flatten()
        stds = np.sqrt(gmm.covariances_.flatten())

        # sort components by mean
        order = np.argsort(means)
        w1, w2 = weights[order]
        m1, m2 = means[order]
        s1, s2 = stds[order]

        # intersection of weighted PDFs
        f = lambda x: w1 * norm.pdf(x, m1, s1) - w2 * norm.pdf(x, m2, s2)

        try:
            importance_threshold = brentq(f, m1, m2)
            threshold_type = "GMM_intersection"
        except ValueError:
            importance_threshold = np.inf   # no clean intersection
            threshold_type = "GMM_no_intersection"

    print()
    # ---------------- apply GMM threshold to dense matrix ----------------
    y_pred_gmm = (y_score >= importance_threshold).astype(int)

    if y_pred_gmm.sum() > 0:
        p_gmm = precision_score(y_true, y_pred_gmm, zero_division=0)
        r_gmm = recall_score(y_true, y_pred_gmm)
        f1_gmm = f1_score(y_true, y_pred_gmm)
    else:
        p_gmm = r_gmm = f1_gmm = 0.0
    post = None
    posterior_cutoff = None
    threshold_type = "GMM_intersection"
    # plot_gmm_diagnostics(
    #     axes[0],
    #     y_score,
    #     gmm,
    #     post,
    #     threshold_type,
    #     best['threshold'] - 0.1,
    #     posterior_cutoff
    # )

    N_RANDOM = 1
    rng = np.random.default_rng(i)

    precisions = []
    recalls = []
    f1s = []

    for _ in range(N_RANDOM):
        y_pred_rand = rng.binomial(1, 0.5, size=len(y_true))

        precisions.append(
            precision_score(y_true, y_pred_rand, zero_division=0)
        )
        recalls.append(
            recall_score(y_true, y_pred_rand)
        )
        f1s.append(
            f1_score(y_true, y_pred_rand)
        )

    rand_precision = np.mean(precisions)
    rand_recall    = np.mean(recalls)
    rand_f1        = np.mean(f1s)

    rand_precision_sd = np.std(precisions)
    rand_recall_sd    = np.std(recalls)
    rand_f1_sd        = np.std(f1s)


    
    # ========================================================
    # Store
    # ========================================================
    metric_rows.append({
        "network": network_path.name,
        "f1_best": best["f1"],
        "precision_best": best["precision"],
        "recall_best": best["recall"],
        "n_edges_best": best["n_edges"],
        "threshold_best": best['threshold'],
        "f1_gmm": f1_gmm,
        "precision_gmm": p_gmm,
        "recall_gmm": r_gmm,
        "n_edges_gmm": int(y_pred_gmm.sum()),
        "gmm_threshold": float(importance_threshold),
        "gmm_mode": threshold_type,
        "f1_random": rand_f1,
        "precision_random": rand_precision,
        "recall_random": rand_recall
    })
grn_boost_and_random_metric_data = pd.DataFrame(metric_rows)
grn_boost_and_random_metric_data.to_csv(f"{path_to_plot_data}/metrics_analysis_data/metrics_grnBoost_random.csv")